# 4. Recipe Generation Module

This module handles recipe generation using multiple models:
- GPT-2 (fine-tuned)
- Llama 3.2 1B (fine-tuned with LoRA)
- Llama 3.1 8B GGUF

**Features:**
- Strengthened prompts with concrete examples
- Format validation for time fields
- LLM-based time estimation fallback

**Usage:**
```python
%run 1_model_loading.ipynb
%run 4_recipe_generation.ipynb
recipe = generate_recipe_with_selected_model(
    ingredient='chicken',
    cuisine='Asian',
    difficulty='beginner',
    model_type=RecipeModelType.LLAMA_1B
)
```

In [1]:
import re
import torch
from typing import Dict, List, Any, Optional

print("✓ Imports loaded")

✓ Imports loaded


## Llama Recipe Generation Functions

In [2]:
def generate_recipe_from_ingredients(model, tokenizer, ingredients_str, cuisine=None):
    """
    Generate recipe from ingredients using fine-tuned Llama model
    STRENGTHENED PROMPT with concrete examples
    """
    cuisine_hint = f" ({cuisine} style)" if cuisine else ""
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. You MUST follow this EXACT format for EVERY recipe.

REQUIRED FORMAT (copy this structure EXACTLY):

# [Recipe Title]

**Prep Time**: 15 minutes
**Cook Time**: 30 minutes  
**Total Time**: 45 minutes
**Servings**: 4

## Ingredients
- ingredient 1
- ingredient 2

## Instructions
1. Step 1
2. Step 2

CRITICAL RULES:
1. ALWAYS include Prep Time, Cook Time, Total Time, Servings (no exceptions!)
2. Time format MUST be: "**Prep Time**: [number] minutes"
3. Use realistic cooking times based on recipe complexity
4. Start immediately with "# " followed by recipe title<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a delicious recipe using these ingredients: {ingredients_str}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.8,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.1
    )
    
    full_output = tokenizer.decode(outputs[0], skip_special_tokens=False)
    response = full_output.split("<|start_header_id|>assistant<|end_header_id|>")[-1]
    response = response.split("<|eot_id|>")[0].strip()
    
    return response

print("✓ Llama recipe generation function defined")

✓ Llama recipe generation function defined


In [3]:
def estimate_times_with_llm(model, tokenizer, recipe_text: str, difficulty: str) -> Dict[str, int]:
    """Use Llama to estimate Prep/Cook/Total time for a recipe"""
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a cooking time expert.<|eot_id|><|start_header_id|>user<|end_header_id|>

Estimate realistic cooking times for this {difficulty} recipe. Reply ONLY with numbers in this format:
Prep: [X]
Cook: [Y]
Total: [Z]

Recipe:
{recipe_text[:300]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Prep: """
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=50,
        temperature=0.3,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = response.split("assistant")[-1].strip()
    
    # Extract numbers
    prep_match = re.search(r'Prep:\s*(\d+)', response, re.IGNORECASE)
    cook_match = re.search(r'Cook:\s*(\d+)', response, re.IGNORECASE)
    total_match = re.search(r'Total:\s*(\d+)', response, re.IGNORECASE)
    
    prep_time = int(prep_match.group(1)) if prep_match else 15
    cook_time = int(cook_match.group(1)) if cook_match else 25
    total_time = int(total_match.group(1)) if total_match else (prep_time + cook_time)
    
    return {"prep": prep_time, "cook": cook_time, "total": total_time}

print("✓ LLM time estimation function defined")

✓ LLM time estimation function defined


In [4]:
def estimate_times_with_gguf(llm, recipe_text: str, difficulty: str) -> Dict[str, int]:
    """Use GGUF Llama to estimate Prep/Cook/Total time for a recipe"""
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a cooking time expert.<|eot_id|><|start_header_id|>user<|end_header_id|>

Estimate realistic cooking times for this {difficulty} recipe. Reply ONLY with numbers in this format:
Prep: [X]
Cook: [Y]
Total: [Z]

Recipe:
{recipe_text[:300]}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Prep: """
    
    output = llm(
        prompt,
        max_tokens=50,
        temperature=0.3,
        top_p=0.9,
        stop=["<|eot_id|>", "\n\n"]
    )
    
    response = output['choices'][0]['text']
    
    # Extract numbers
    prep_match = re.search(r'Prep:\s*(\d+)', response, re.IGNORECASE)
    cook_match = re.search(r'Cook:\s*(\d+)', response, re.IGNORECASE)
    total_match = re.search(r'Total:\s*(\d+)', response, re.IGNORECASE)
    
    prep_time = int(prep_match.group(1)) if prep_match else 15
    cook_time = int(cook_match.group(1)) if cook_match else 25
    total_time = int(total_match.group(1)) if total_match else (prep_time + cook_time)
    
    return {"prep": prep_time, "cook": cook_time, "total": total_time}

print("✓ GGUF time estimation function defined")

✓ GGUF time estimation function defined


In [5]:
def validate_time_format(response: str) -> Dict[str, bool]:
    """
    Validate that time fields exist and are in correct format
    Returns dict with validation results
    """
    validation = {
        'has_prep_time': False,
        'has_cook_time': False,
        'has_total_time': False,
        'has_servings': False,
        'prep_format_correct': False,
        'cook_format_correct': False,
        'total_format_correct': False,
        'servings_format_correct': False
    }
    
    # Check for existence and correct format: "**Prep Time**: [number] minutes"
    prep_pattern = r'\*\*Prep Time\*\*:\s*(\d+)\s*minutes?'
    cook_pattern = r'\*\*Cook Time\*\*:\s*(\d+)\s*minutes?'
    total_pattern = r'\*\*Total Time\*\*:\s*(\d+)\s*minutes?'
    servings_pattern = r'\*\*Servings\*\*:\s*(\d+)'
    
    prep_match = re.search(prep_pattern, response, re.IGNORECASE)
    cook_match = re.search(cook_pattern, response, re.IGNORECASE)
    total_match = re.search(total_pattern, response, re.IGNORECASE)
    servings_match = re.search(servings_pattern, response, re.IGNORECASE)
    
    if prep_match:
        validation['has_prep_time'] = True
        validation['prep_format_correct'] = True
    
    if cook_match:
        validation['has_cook_time'] = True
        validation['cook_format_correct'] = True
    
    if total_match:
        validation['has_total_time'] = True
        validation['total_format_correct'] = True
    
    if servings_match:
        validation['has_servings'] = True
        validation['servings_format_correct'] = True
    
    return validation

print("✓ Format validation function defined")

✓ Format validation function defined


In [6]:
def ensure_time_fields_with_llm(response: str, difficulty: str, model=None, tokenizer=None, llm=None) -> str:
    """
    Ensure recipe has Prep Time, Cook Time, Total Time, Servings with correct format
    Only uses LLM estimation if fields are completely missing
    """
    # First, validate the current format
    validation = validate_time_format(response)
    
    # If all fields exist in correct format, return as-is (best case - no fallback needed!)
    all_valid = (validation['prep_format_correct'] and 
                 validation['cook_format_correct'] and 
                 validation['total_format_correct'] and 
                 validation['servings_format_correct'])
    
    if all_valid:
        return response
    
    # If fields are missing, use LLM to estimate times
    if model and tokenizer:
        times = estimate_times_with_llm(model, tokenizer, response, difficulty)
    elif llm:
        times = estimate_times_with_gguf(llm, response, difficulty)
    else:
        # Ultimate fallback (shouldn't happen)
        times = {"prep": 15, "cook": 25, "total": 40}
    
    # Insert missing fields after title (using CORRECT format)
    lines = response.split('\n')
    new_lines = []
    inserted = False
    
    for line in lines:
        new_lines.append(line)
        
        # Insert after first # heading
        if not inserted and line.startswith('#') and not line.startswith('##'):
            new_lines.append('')
            if not validation['prep_format_correct']:
                new_lines.append(f'**Prep Time**: {times["prep"]} minutes')
            if not validation['cook_format_correct']:
                new_lines.append(f'**Cook Time**: {times["cook"]} minutes')
            if not validation['total_format_correct']:
                new_lines.append(f'**Total Time**: {times["total"]} minutes')
            if not validation['servings_format_correct']:
                new_lines.append(f'**Servings**: 4')
            new_lines.append('')
            inserted = True
    
    return '\n'.join(new_lines)

print("✓ Time field enforcement function defined")

✓ Time field enforcement function defined


In [7]:
def parse_llama_recipe_output(response: str, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None, model=None, tokenizer=None) -> Dict[str, Any]:
    """
    Format Llama output - extract title, time, servings from content
    Uses Llama to estimate missing times only if needed
    """
    # Ensure time fields exist with correct format (minimal fallback)
    response = ensure_time_fields_with_llm(response, difficulty, model, tokenizer, llm=None)
    
    lines = response.strip().split('\n')
    
    # Extract title (first # heading)
    title = "Recipe"
    for line in lines:
        if line.startswith('#') and not line.startswith('##'):
            title = line.strip('# ').strip()
            break
    
    # Extract cooking time from Markdown content
    cooking_time = 30  # default
    time_patterns = [
        r'(?:Cooking Time|Cook Time|Total Time|Time)[\s:*]+(\d+)[\s-]*(?:min|minute)',
        r'\*\*(?:Cooking Time|Cook Time|Total Time|Time)\*\*[\s:]+(\d+)',
        r'(\d+)[\s-]*(?:min|minute)(?:ute)?s?\s+(?:cooking|cook|total)',
    ]
    
    response_lower = response.lower()
    for pattern in time_patterns:
        match = re.search(pattern, response_lower, re.IGNORECASE)
        if match:
            cooking_time = int(match.group(1))
            break
    
    # Extract servings: priority: nutrition data > Markdown content > default
    servings = 4  # default
    
    # Priority 1: Use nutrition data if available
    if nutrition_data and 'servings' in nutrition_data:
        servings = nutrition_data['servings']
    else:
        # Priority 2: Extract from Markdown content
        servings_patterns = [
            r'(?:Servings|Serves|Yield)[\s:*]+(\d+)',
            r'\*\*(?:Servings|Serves|Yield)\*\*[\s:]+(\d+)',
            r'(?:Makes|Yields)\s+(\d+)\s+(?:serving|portion)',
        ]
        
        for pattern in servings_patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                servings = int(match.group(1))
                break
    
    # Keep complete Markdown content
    return {
        "ingredient": ingredient,
        "recipe_title": title,
        "cuisine": cuisine,
        "difficulty": difficulty,
        "cooking_time_minutes": cooking_time,
        "servings": servings,
        "raw_markdown": response,
        "ingredients": [ingredient],
        "instructions": [response]
    }

print("✓ Llama output parsing function defined")

✓ Llama output parsing function defined


In [8]:
def generate_recipe_llama(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None) -> Dict[str, Any]:
    """Generate recipe using Llama model (fine-tuned)"""
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    
    # Generate recipe
    response = generate_recipe_from_ingredients(model, tokenizer, ingredient, cuisine)
    
    # Parse and ensure times (pass model for time estimation if needed)
    return parse_llama_recipe_output(response, ingredient, cuisine, difficulty, nutrition_data, model, tokenizer)

print("✓ Llama recipe generation wrapper defined")

✓ Llama recipe generation wrapper defined


## GGUF Recipe Generation Functions

In [9]:
def parse_gguf_recipe_output(response: str, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None, llm=None) -> Dict[str, Any]:
    """
    Format GGUF output - extract title, time, servings from content
    Uses Llama to estimate missing times only if needed
    """
    # Ensure time fields exist with correct format (minimal fallback)
    response = ensure_time_fields_with_llm(response, difficulty, model=None, tokenizer=None, llm=llm)
    
    lines = response.strip().split('\n')
    
    # Extract title (first # heading)
    title = "Recipe"
    for line in lines:
        if line.startswith('#') and not line.startswith('##'):
            title = line.strip('# ').strip()
            break
    
    # Extract cooking time from Markdown content
    cooking_time = 30  # default
    time_patterns = [
        r'(?:Cooking Time|Cook Time|Total Time|Time)[\s:*]+(\d+)[\s-]*(?:min|minute)',
        r'\*\*(?:Cooking Time|Cook Time|Total Time|Time)\*\*[\s:]+(\d+)',
        r'(\d+)[\s-]*(?:min|minute)(?:ute)?s?\s+(?:cooking|cook|total)',
    ]
    
    response_lower = response.lower()
    for pattern in time_patterns:
        match = re.search(pattern, response_lower, re.IGNORECASE)
        if match:
            cooking_time = int(match.group(1))
            break
    
    # Extract servings: priority: nutrition data > Markdown content > default
    servings = 4  # default
    
    if nutrition_data and 'servings' in nutrition_data:
        servings = nutrition_data['servings']
    else:
        servings_patterns = [
            r'(?:Servings|Serves|Yield)[\s:*]+(\d+)',
            r'\*\*(?:Servings|Serves|Yield)\*\*[\s:]+(\d+)',
            r'(?:Makes|Yields)\s+(\d+)\s+(?:serving|portion)',
        ]
        
        for pattern in servings_patterns:
            match = re.search(pattern, response, re.IGNORECASE)
            if match:
                servings = int(match.group(1))
                break
    
    return {
        "ingredient": ingredient,
        "recipe_title": title,
        "cuisine": cuisine,
        "difficulty": difficulty,
        "cooking_time_minutes": cooking_time,
        "servings": servings,
        "raw_markdown": response,
        "ingredients": [ingredient],
        "instructions": [response]
    }

print("✓ GGUF output parsing function defined")

✓ GGUF output parsing function defined


In [10]:
def generate_recipe_gguf(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str, nutrition_data: dict = None) -> Dict[str, Any]:
    """Generate recipe using GGUF model (Llama 3.1 8B)"""
    llm = model_dict["model"]
    cuisine_hint = f" ({cuisine} style)" if cuisine and cuisine != "any" else ""
    
    prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are an expert chef assistant. You MUST follow this EXACT format for EVERY recipe.

REQUIRED FORMAT (copy this structure EXACTLY):

# [Recipe Title]

**Prep Time**: 15 minutes
**Cook Time**: 30 minutes
**Total Time**: 45 minutes
**Servings**: 4

## Ingredients
- ingredient 1
- ingredient 2

## Instructions
1. Step 1
2. Step 2

CRITICAL RULES:
1. ALWAYS include Prep Time, Cook Time, Total Time, Servings (no exceptions!)
2. Time format MUST be: "**Prep Time**: [number] minutes"
3. Use realistic cooking times based on recipe complexity
4. Start immediately with "# " followed by recipe title<|eot_id|><|start_header_id|>user<|end_header_id|>

Create a {difficulty} recipe using: {ingredient}{cuisine_hint}.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

# """
    
    output = llm(
        prompt,
        max_tokens=512,
        temperature=0.7,
        top_p=0.9,
        stop=["<|eot_id|>"]
    )
    
    response = output['choices'][0]['text']
    
    # Parse and ensure times (pass llm for time estimation if needed)
    return parse_gguf_recipe_output(response, ingredient, cuisine, difficulty, nutrition_data, llm)

print("✓ GGUF recipe generation function defined")

✓ GGUF recipe generation function defined


## GPT-2 Recipe Generation (Simplified)

In [11]:
def generate_recipe_gpt2(model_dict: Dict, ingredient: str, cuisine: str, difficulty: str) -> Dict[str, Any]:
    """Generate recipe using GPT-2 (simplified - basic generation only)"""
    model = model_dict["model"]
    tokenizer = model_dict["tokenizer"]
    device = model_dict["device"]
    
    prompt = f"""<INGREDIENT> {ingredient}
<CUISINE> {cuisine}
<TITLE> """
    
    input_ids = tokenizer.encode(prompt, return_tensors='pt').to(device)
    
    with torch.no_grad():
        output = model.generate(
            input_ids,
            max_length=350,
            temperature=0.5,
            top_k=50,
            top_p=0.95,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=2
        )
    
    generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
    
    # Simple parsing for GPT-2 output
    return {
        "ingredient": ingredient,
        "recipe_title": f"{cuisine} Style {ingredient.title()}",
        "cuisine": cuisine.lower(),
        "difficulty": difficulty.lower(),
        "cooking_time_minutes": 30,
        "servings": 4,
        "raw_text": generated_text,
        "ingredients": [ingredient],
        "instructions": [generated_text]
    }

print("✓ GPT-2 recipe generation function defined (simplified)")

✓ GPT-2 recipe generation function defined (simplified)


## Model Router

In [12]:
def generate_recipe_with_selected_model(
    ingredient: str,
    cuisine: str = "any",
    difficulty: str = "medium",
    model_type = None,  # RecipeModelType
    nutrition_data: dict = None
) -> Dict[str, Any]:
    """
    Generate recipe using selected model
    
    This is the main entry point for recipe generation.
    It routes to the appropriate generation function based on model type.
    
    NOTE: Requires load_recipe_model() from 1_model_loading.ipynb
    """
    # Import RecipeModelType if not available
    try:
        from enum import Enum
        if model_type is None:
            # Use CURRENT_MODEL_TYPE if available
            model_type = globals().get('CURRENT_MODEL_TYPE')
    except:
        pass
    
    # Load model (or use cached) - requires 1_model_loading.ipynb
    model_dict = load_recipe_model(model_type)
    
    # Route to appropriate generation function
    if model_dict["type"] == "gpt2":
        return generate_recipe_gpt2(model_dict, ingredient, cuisine, difficulty)
    elif model_dict["type"] == "llama":
        return generate_recipe_llama(model_dict, ingredient, cuisine, difficulty, nutrition_data)
    elif model_dict["type"] == "gguf":
        return generate_recipe_gguf(model_dict, ingredient, cuisine, difficulty, nutrition_data)
    else:
        raise ValueError(f"Unknown model type: {model_dict.get('type')}")

print("✓ Model router defined")

✓ Model router defined


## Helper: Generate Diverse Prompts

In [13]:
def generate_diverse_prompts(ingredient: str, num_recipes: int = 5) -> List[Dict]:
    """Generate diverse cuisine prompts"""
    configurations = [
        {'cuisine': 'Asian', 'difficulty': 'beginner'},
        {'cuisine': 'Western', 'difficulty': 'beginner'},
        {'cuisine': 'Fusion', 'difficulty': 'intermediate'},
        {'cuisine': 'Mediterranean', 'difficulty': 'beginner'},
        {'cuisine': 'any', 'difficulty': 'intermediate'},
    ]
    
    prompts = []
    for i in range(min(num_recipes, len(configurations))):
        config = configurations[i]
        prompts.append({
            'ingredient': ingredient,
            'cuisine': config['cuisine'],
            'difficulty': config['difficulty'],
            'recipe_index': i + 1
        })
    
    return prompts

print("✓ Diverse prompts generator defined")

✓ Diverse prompts generator defined


---

**Module exports:**
- `generate_recipe_with_selected_model()` (main function)
- `generate_recipe_llama()` (Llama generation)
- `generate_recipe_gguf()` (GGUF generation)
- `generate_recipe_gpt2()` (GPT-2 generation)
- `generate_diverse_prompts()` (helper)
- `validate_time_format()` (validation)
- `ensure_time_fields_with_llm()` (fallback)